# 第3回：LLMを動かして理解する③

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session03/session03_llm_basics.ipynb)

このノートブックでは LLMの動作を手を動かしながら体験する。


## 準備
---

> **GPU ランタイムに切り替えて使用する（T4 以上推奨）。**  
> Colab メニュー → ランタイム → ランタイムのタイプを変更 → T4 GPU  
> GPUランタイムが利用できない場合はCPUでも可（ただし遅い）

### パッケージのインストール

In [ ]:
# パッケージのインストール
!pip install -q openai
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!uv pip install "vllm==0.19.1" --torch-backend=cu129 -q
#!uv pip install --system "vllm==0.19.1" -q

---
## A. vLLM

- GPU を使った高速 LLM 推論フレームワーク
- PagedAttention により大量リクエストを効率処理
- OpenAI 互換 API サーバーもサポート

> **GPU ランタイムが必要（T4 以上）**

### A-1. Python API

In [ ]:
from vllm.distributed import cleanup_dist_env_and_memory
import gc
import torch


def cleanup_vllm_memory():
    cleanup_dist_env_and_memory()
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

- ベースモデル（`Qwen/Qwen2.5-0.5B`）

In [ ]:
from vllm import LLM, SamplingParams

llm_base = LLM(model='Qwen/Qwen2.5-0.5B', max_model_len=512, dtype='auto', gpu_memory_utilization=0.8)

sampling_params = SamplingParams(temperature=0.5, max_tokens=100)
outputs = llm_base.generate(['東京は日本の首都であり、'], sampling_params)

# メモリ解放
del llm_base
cleanup_vllm_memory()

print('[vLLM / ベースモデル / テキスト補完]')
print(outputs[0].outputs[0].text)

- チャットモデル（`Qwen/Qwen2.5-0.5B-Instruct`）

In [ ]:
llm_chat = LLM(model='Qwen/Qwen2.5-0.5B-Instruct', max_model_len=512, dtype='auto', gpu_memory_utilization=0.8)

conversation = [
    {'role': 'system', 'content': '簡潔に答えてください。'},
    {'role': 'user',   'content': 'AIエージェントとは何ですか？'},
]
sampling_params = SamplingParams(temperature=0.0, max_tokens=150)
outputs = llm_chat.chat(messages=[conversation], sampling_params=sampling_params)

# メモリ解放
del llm_chat
cleanup_vllm_memory()

print('[vLLM / チャットモデル / 指示への回答]')
print(outputs[0].outputs[0].text)

### A-2. API サーバー（OpenAI 互換）

- ベースモデル（`Qwen/Qwen2.5-0.5B`）

  ターミナルを開き、次のコマンドでサーバーを起動する。

  ```bash
  python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-0.5B \
    --port 8000 \
    --max-model-len 512 \
    --dtype auto
  ```

  `Application startup complete` のようなログが出たら準備完了。下のセルからクライアントで接続します（サーバーは別ターミナルで起動したまま使い、終了時は `Ctrl+C` で停止）。

  > ベースモデルなので、クライアント側は `completions.create`（プロンプトの続きを生成）を使う。

In [ ]:
from openai import OpenAI
import json

vllm_client = OpenAI(base_url='http://localhost:8000/v1', api_key='dummy')

response = vllm_client.completions.create(
    model='Qwen/Qwen2.5-0.5B',
    prompt='東京は日本の首都であり、',
    max_tokens=150,
)

print('[vLLM OpenAI 互換サーバー / 非チャット]')
print(response.model_dump_json(indent=2))


- チャットモデル（`Qwen/Qwen2.5-0.5B-Instruct`）

  ターミナルを開き、次のコマンドでサーバーを起動する。

  ```bash
  python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --port 8080 \
    --max-model-len 512 \
    --dtype auto
  ```

  `Application startup complete` のようなログが出たら準備完了。下のセルからクライアントで接続する（サーバーは別ターミナルで起動したまま使い、終了時は `Ctrl+C` で停止）。

In [ ]:
from openai import OpenAI

vllm_client = OpenAI(base_url='http://localhost:8080/v1', api_key='dummy')

response = vllm_client.chat.completions.create(
    model='Qwen/Qwen2.5-0.5B-Instruct',
    messages=[
        {'role': 'system', 'content': '簡潔に答えてください。'},
        {'role': 'user',   'content': 'AIエージェントとは何ですか？'},
    ],
    max_tokens=150,
)

print('[vLLM OpenAI 互換サーバー / チャット]')
print(response.model_dump_json(indent=2))

---
## B. Ollama

バックエンドに `llama.cpp` を使用。  
**CLI**実行と **REST API サーバー**に対応

バックグラウンドで起動
```bash
nohup ollama serve > ollama.log 2>&1 &
```

### B-1. CLI実行

ターミナルを開き、`ollama run <model> "<prompt>"` を実行

```
ollama run qwen2.5:0.5b 'AIエージェントとは何ですか？簡潔に答えてください。' --nowordwrap
```

### B-2. API

| エンドポイント | 用途 |
|---------------|------|
| `POST /api/generate` | テキスト補完（`prompt` キー） |
| `POST /api/chat` | チャット補完（`messages` キー、OpenAI 互換） |

APIサーバーとして使用する場合は事前にモデルをダウンロードしておく

```bash
ollama pull qwen3:0.6b
```

In [ ]:
# サーバー疎通確認（ターミナルでセットアップ済みの前提）
import requests

response = requests.get('http://localhost:11434/api/tags')
print('[Ollama サーバー疎通OK / インストール済みモデル]')
for model in response.json().get('models', []):
    print(' -', model['name'])

- テキスト補完API

In [ ]:
import requests
import json

# /api/generate：テキスト補完
payload = {
    'model': 'qwen3:0.6b', # ※Ollamaのqwen3:0.6bはチャット形式のインストラクション・モデルであるがテキスト補完でも動く
    'prompt': '東京は日本の首都であり、',
    'stream': False,
    'think': False,  # qwen3 は thinking モデルのため、<think>...</think> の出力を抑止する
    'options': {'temperature': 0.0, 'num_predict': 60},
}
response = requests.post('http://localhost:11434/api/generate', json=payload)
print('[Ollama /api/generate / テキスト補完]')
print(json.dumps(response.json(), ensure_ascii=False, indent=2))

- チャット補完API

In [ ]:
# /api/chat：チャット補完（OpenAI 互換メッセージ形式）
payload = {
    'model': 'qwen3:0.6b',
    'messages': [
        {'role': 'system', 'content': '簡潔に答えてください。'},
        {'role': 'user',   'content': 'AIエージェントとは何ですか？'},
    ],
    'stream': False,
}
response = requests.post('http://localhost:11434/api/chat', json=payload)
print('[Ollama /api/chat / チャット補完]')
print(json.dumps(response.json(), ensure_ascii=False, indent=2))

### B-3. OpenAI 互換 API

| エンドポイント | 用途 |
|---------------|------|
| `POST /v1/completions` | テキスト補完（`prompt` キー） |
| `POST /v1/chat/completions` | チャット補完（`messages` キー） |

- テキスト補完API

In [ ]:
import json
from openai import OpenAI

ollama_openai_client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
response = ollama_openai_client.completions.create(
    model='qwen3:0.6b',
    prompt='東京は日本の首都であり、',
    temperature=0.0,
    max_tokens=80,
)

print('[Ollama OpenAI互換 /v1/completions]')
print(response.model_dump_json(indent=2))

- チャット補完API

In [ ]:
response = ollama_openai_client.chat.completions.create(
    model='qwen3:0.6b',
    messages=[
        {'role': 'system', 'content': '簡潔に答えてください。'},
        {'role': 'user', 'content': '東京の有名な観光地を2つ挙げてください。'},
    ],
    temperature=0.0,
    max_tokens=80,
)

print('[Ollama OpenAI互換 /v1/chat/completions]')
print(response.model_dump_json(indent=2))